# Práctica 24 · Clase 27
## PySpark: Agregaciones, joins, ventanas y escritura de datos

**Alumno:** Héctor Miguel Tudela

Ejercicios resueltos de la clase. La celda de configuración se ejecuta **siempre la primera** (y de nuevo tras reiniciar el kernel). Detecta sola si se está en Spark local, en el clúster Docker o en Databricks.

## Celda de configuración

In [1]:
import os
from pathlib import Path

# 1. Detectar el entorno
if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    ENTORNO = "databricks"
elif Path("/opt/datos").is_dir():
    ENTORNO = "docker"            # contenedor jupyter del Tutorial 2
else:
    ENTORNO = "local"             # Spark local del Tutorial 1

# 2. Carpeta base de los datos (fuera de Databricks)
if ENTORNO == "docker":
    BASE = "/opt/datos"
elif os.name == "nt":
    BASE = "C:/SparkLocal/workspace/datos"
else:
    BASE = str(Path.home() / "spark_datos")                    # por si alguien usa Linux o macOS

# 3. SparkSession: en Databricks ya existe; en los otros dos se crea con el catálogo de Hive
if ENTORNO != "databricks":
    from pyspark.sql import SparkSession

    builder = SparkSession.builder.appName("clase_27").enableHiveSupport()
    if ENTORNO == "local":
        builder = (
            builder.master("local[*]")
            .config("spark.sql.warehouse.dir", f"{BASE}/warehouse")
            .config("spark.hadoop.javax.jdo.option.ConnectionURL",
                    f"jdbc:derby:;databaseName={BASE}/metastore/metastore_db;create=true")
        )
        if os.name == "nt":       # Windows: caché de JARs y carpeta temporal de Hive del Tutorial 1
            builder = (
                builder.config("spark.jars.ivy", "C:/SparkLocal/tools/cache/ivy2")
                .config("spark.hadoop.hive.exec.scratchdir", "C:/SparkLocal/tmp/hive")
            )
    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

# 4. Base de datos de las tablas (BD) y carpeta de ficheros de la clase (RUTA)
if ENTORNO == "databricks":
    spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.clase_27")
    spark.sql("CREATE VOLUME IF NOT EXISTS workspace.clase_27.datos")
    BD = "workspace.clase_27"
    RUTA = "/Volumes/workspace/clase_27/datos"
else:
    spark.sql("CREATE DATABASE IF NOT EXISTS clase_27")
    BD = "clase_27"
    RUTA = f"{BASE}/clase_27"

Path(RUTA).mkdir(parents=True, exist_ok=True)

print("Entorno:", ENTORNO)
print("Versión de Spark:", spark.version)
print("Carpeta de datos:", RUTA)
print("Base de datos de las tablas:", BD)
if ENTORNO != "databricks":
    print("Catálogo:", spark.conf.get("spark.sql.catalogImplementation"))

Entorno: local
Versión de Spark: 4.1.4
Carpeta de datos: /root/spark_datos/clase_27
Base de datos de las tablas: clase_27
Catálogo: hive


## Sección 1. Agregaciones sobre todo el DataFrame

### Ejemplo 1.1 · Resumen del día de una cadena de ferreterías

**Enunciado.** Una cadena de ferreterías de Aragón quiere un resumen de los tickets del día: número de tickets, unidades vendidas, facturación, ticket medio, ticket más bajo y más alto, y cuántos productos distintos se han vendido. Después necesita la facturación como número de Python para enviarla en un informe, y cuántas tiendas han vendido algo.

In [2]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

ventas = spark.createDataFrame(
    [("T-001", "Zaragoza", "Taladro percutor", 1, 89.90),
     ("T-002", "Zaragoza", "Tornillería surtida", 4, 6.50),
     ("T-003", "Huesca", "Taladro percutor", 2, 89.90),
     ("T-004", "Teruel", "Escalera de aluminio", 1, 74.00),
     ("T-005", "Huesca", "Pintura plástica 15 L", 3, 42.75),
     ("T-006", "Zaragoza", "Escalera de aluminio", 1, 74.00)],
    "ticket STRING, tienda STRING, producto STRING, unidades INT, precio DOUBLE",
).withColumn("importe", F.round(col("unidades") * col("precio"), 2))

resumen = ventas.agg(
    F.count("*").alias("tickets"),
    F.sum("unidades").alias("unidades"),
    F.round(F.sum("importe"), 2).alias("facturacion"),
    F.round(F.avg("importe"), 2).alias("ticket_medio"),
    F.min("importe").alias("ticket_min"),
    F.max("importe").alias("ticket_max"),
    F.countDistinct("producto").alias("productos"),
)
resumen.show()

fila = resumen.first()
print("Facturación total:", fila.facturacion, "€")
print("Tiendas con ventas:", ventas.select(F.countDistinct("tienda")).first()[0])

+-------+--------+-----------+------------+----------+----------+---------+
|tickets|unidades|facturacion|ticket_medio|ticket_min|ticket_max|productos|
+-------+--------+-----------+------------+----------+----------+---------+
|      6|      12|     571.95|       95.33|      26.0|     179.8|        4|
+-------+--------+-----------+------------+----------+----------+---------+



Facturación total: 571.95 €


Tiendas con ventas: 3


### Ejemplo 1.2 · Los nulos en las agregaciones: encuestas de satisfacción

**Enunciado.** Una cadena hotelera envía una encuesta de satisfacción (puntuación de 0 a 10) a los clientes al terminar su estancia. No todos responden: las encuestas sin respuesta tienen la puntuación a `NULL`. Dirección pregunta cuál es la puntuación media, y el equipo de datos comprueba cómo afectan los nulos a cada función.

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

encuestas = spark.createDataFrame(
    [("E-1", "Madrid", 9), ("E-2", "Madrid", None), ("E-3", "Sevilla", 7),
     ("E-4", "Sevilla", None), ("E-5", "Bilbao", 10), ("E-6", "Madrid", 6)],
    "encuesta STRING, hotel STRING, puntuacion INT",
)

encuestas.agg(
    F.count("*").alias("enviadas"),
    F.count("puntuacion").alias("respondidas"),
    F.sum("puntuacion").alias("suma_puntos"),
    F.round(F.avg("puntuacion"), 2).alias("media_respondidas"),
    F.round(F.avg(F.coalesce(col("puntuacion"), F.lit(0))), 2).alias("media_nulos_como_0"),
).show()

+--------+-----------+-----------+-----------------+------------------+
|enviadas|respondidas|suma_puntos|media_respondidas|media_nulos_como_0|
+--------+-----------+-----------+-----------------+------------------+
|       6|          4|         32|              8.0|              5.33|
+--------+-----------+-----------+-----------------+------------------+



## Sección 2. Agrupar con `groupBy()`

### Ejemplo 2.1 · Facturación por canal de una tienda de cosmética

**Enunciado.** Una marca de cosmética vende por su web, por su app y en sus tiendas físicas. El equipo de marketing quiere saber, para cada canal, cuántos pedidos hay, cuánto se ha facturado y cuál es el ticket medio, ordenado de mayor a menor facturación. También quiere el número de pedidos por canal en orden alfabético, con el atajo `count()`.

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

pedidos = spark.createDataFrame(
    [("P-01", "web", 45.90), ("P-02", "app", 120.00), ("P-03", "web", 32.50),
     ("P-04", "tienda", 210.00), ("P-05", "app", 58.40), ("P-06", "web", 99.99),
     ("P-07", "app", 15.00), ("P-08", "tienda", 64.30)],
    "pedido STRING, canal STRING, importe DOUBLE",
)

por_canal = (
    pedidos
    .groupBy("canal")
    .agg(
        F.count("*").alias("pedidos"),
        F.round(F.sum("importe"), 2).alias("facturacion"),
        F.round(F.avg("importe"), 2).alias("ticket_medio"),
    )
    .orderBy(col("facturacion").desc())
)
por_canal.show()

print("Columnas del resultado:", por_canal.columns)
pedidos.groupBy("canal").count().orderBy("canal").show()

+------+-------+-----------+------------+
| canal|pedidos|facturacion|ticket_medio|
+------+-------+-----------+------------+
|tienda|      2|      274.3|      137.15|
|   app|      3|      193.4|       64.47|
|   web|      3|     178.39|       59.46|
+------+-------+-----------+------------+

Columnas del resultado: ['canal', 'pedidos', 'facturacion', 'ticket_medio']


+------+-----+
| canal|count|
+------+-----+
|   app|    3|
|tienda|    2|
|   web|    3|
+------+-----+



### Ejemplo 2.2 · Agrupar por varias columnas, filtrar antes y después

**Enunciado.** Un distribuidor de material de oficina analiza sus líneas de venta por región y categoría. Las líneas **devueltas** no deben contar. Del resultado, solo interesan los grupos que facturan **800 € o más**.

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

lineas = spark.createDataFrame(
    [("Norte", "Papel", 420.0, False), ("Norte", "Papel", 380.0, False),
     ("Norte", "Mobiliario", 1350.0, False), ("Sur", "Papel", 260.0, True),
     ("Sur", "Mobiliario", 890.0, False), ("Sur", "Mobiliario", 640.0, False),
     ("Este", "Tecnología", 2100.0, False), ("Este", "Papel", 95.0, False),
     ("Este", "Tecnología", 450.0, True)],
    "region STRING, categoria STRING, importe DOUBLE, devuelto BOOLEAN",
)

resultado = (
    lineas
    .filter(~col("devuelto"))                              # antes de agrupar (WHERE)
    .groupBy("region", "categoria")
    .agg(F.count("*").alias("lineas"), F.sum("importe").alias("facturacion"))
    .filter(col("facturacion") >= 800)                      # después de agrupar (HAVING)
    .orderBy("region", "categoria")
)
resultado.show()

+------+----------+------+-----------+
|region| categoria|lineas|facturacion|
+------+----------+------+-----------+
|  Este|Tecnología|     1|     2100.0|
| Norte|Mobiliario|     1|     1350.0|
| Norte|     Papel|     2|      800.0|
|   Sur|Mobiliario|     2|     1530.0|
+------+----------+------+-----------+



### Ejemplo 2.3 · La cartera de productos de cada cliente de un banco

**Enunciado.** El área comercial de un banco quiere ver, para cada cliente, qué productos tiene contratados, para detectar a los que tienen **tres o más productos distintos** y ofrecerles un plan de vinculación. Un mismo cliente puede tener dos tarjetas del mismo tipo.

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

contratos = spark.createDataFrame(
    [("CL-1", "cuenta"), ("CL-1", "tarjeta"), ("CL-1", "hipoteca"), ("CL-1", "tarjeta"),
     ("CL-2", "cuenta"), ("CL-3", "cuenta"), ("CL-3", "fondo"), ("CL-3", "tarjeta")],
    "cliente STRING, producto STRING",
)

carteras = (
    contratos
    .groupBy("cliente")
    .agg(
        F.count("*").alias("contratos"),
        F.array_sort(F.collect_list("producto")).alias("todos"),
        F.array_sort(F.collect_set("producto")).alias("distintos"),
    )
    .withColumn("num_distintos", F.size(col("distintos")))
    .orderBy("cliente")
)
carteras.show(truncate=False)
print("Clientes con 3 o más productos:",
      [f.cliente for f in carteras.filter(col("num_distintos") >= 3).collect()])

+-------+---------+------------------------------------+---------------------------+-------------+
|cliente|contratos|todos                               |distintos                  |num_distintos|
+-------+---------+------------------------------------+---------------------------+-------------+
|CL-1   |4        |[cuenta, hipoteca, tarjeta, tarjeta]|[cuenta, hipoteca, tarjeta]|3            |
|CL-2   |1        |[cuenta]                            |[cuenta]                   |1            |
|CL-3   |3        |[cuenta, fondo, tarjeta]            |[cuenta, fondo, tarjeta]   |3            |
+-------+---------+------------------------------------+---------------------------+-------------+



Clientes con 3 o más productos: ['CL-1', 'CL-3']


## Sección 3. Agregaciones condicionales y tablas dinámicas

### Ejemplo 3.1 · Informe de tickets de soporte por equipo

**Enunciado.** El departamento de sistemas de una empresa de distribución tiene tres equipos de soporte. Para la reunión semanal necesita, **en una sola tabla**: total de tickets, tickets abiertos, tickets críticos, horas medias de resolución **de los cerrados** y porcentaje de tickets resueltos.

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

tickets = spark.createDataFrame(
    [("S-01", "Redes", "cerrada", "alta", 4.0), ("S-02", "Redes", "abierta", "critica", None),
     ("S-03", "Redes", "cerrada", "media", 10.0), ("S-04", "Correo", "cerrada", "baja", 2.0),
     ("S-05", "Correo", "abierta", "media", None), ("S-06", "Correo", "abierta", "critica", None),
     ("S-07", "ERP", "cerrada", "critica", 18.0), ("S-08", "ERP", "cerrada", "alta", 6.0)],
    "ticket STRING, equipo STRING, estado STRING, prioridad STRING, horas_resolucion DOUBLE",
)

informe = (
    tickets
    .groupBy("equipo")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.when(col("estado") == "abierta", 1).otherwise(0)).alias("abiertas"),
        F.sum(F.when(col("prioridad") == "critica", 1).otherwise(0)).alias("criticas"),
        F.round(F.avg(F.when(col("estado") == "cerrada", col("horas_resolucion"))), 1)
         .alias("horas_medias_cierre"),
    )
    .withColumn("pct_resueltas", F.round((col("total") - col("abiertas")) / col("total") * 100, 1))
    .orderBy("equipo")
)
informe.show()

+------+-----+--------+--------+-------------------+-------------+
|equipo|total|abiertas|criticas|horas_medias_cierre|pct_resueltas|
+------+-----+--------+--------+-------------------+-------------+
|Correo|    3|       2|       1|                2.0|         33.3|
|   ERP|    2|       0|       1|               12.0|        100.0|
| Redes|    3|       1|       1|                7.0|         66.7|
+------+-----+--------+--------+-------------------+-------------+



### Ejemplo 3.2 · Ventas trimestrales de una cadena de ópticas

**Enunciado.** Una cadena de ópticas quiere un cuadro con una fila por tienda y una columna por trimestre, como el que prepararía en una hoja de cálculo. La Óptica Playa abrió en el segundo trimestre y cerró por obras en el cuarto, y la Óptica Norte tiene dos registros en el primer trimestre. Después hay que rellenar los huecos con 0 y añadir el total anual.

In [8]:
from pyspark.sql import functions as F

ventas = spark.createDataFrame(
    [("Óptica Centro", "T1", 12400.0), ("Óptica Centro", "T2", 15100.0),
     ("Óptica Centro", "T3", 9800.0), ("Óptica Centro", "T4", 16900.0),
     ("Óptica Playa", "T2", 8300.0), ("Óptica Playa", "T3", 21500.0),
     ("Óptica Norte", "T1", 7600.0), ("Óptica Norte", "T1", 1200.0),
     ("Óptica Norte", "T4", 9900.0)],
    "tienda STRING, trimestre STRING, importe DOUBLE",
)

tabla = (
    ventas
    .groupBy("tienda")
    .pivot("trimestre", ["T1", "T2", "T3", "T4"])
    .agg(F.sum("importe"))
    .orderBy("tienda")
)
tabla.show()

tabla.fillna(0).withColumn("anual", F.col("T1") + F.col("T2") + F.col("T3") + F.col("T4")).show()

+-------------+-------+-------+-------+-------+
|       tienda|     T1|     T2|     T3|     T4|
+-------------+-------+-------+-------+-------+
|Óptica Centro|12400.0|15100.0| 9800.0|16900.0|
| Óptica Norte| 8800.0|   NULL|   NULL| 9900.0|
| Óptica Playa|   NULL| 8300.0|21500.0|   NULL|
+-------------+-------+-------+-------+-------+



+-------------+-------+-------+-------+-------+-------+
|       tienda|     T1|     T2|     T3|     T4|  anual|
+-------------+-------+-------+-------+-------+-------+
|Óptica Centro|12400.0|15100.0| 9800.0|16900.0|54200.0|
| Óptica Norte| 8800.0|    0.0|    0.0| 9900.0|18700.0|
| Óptica Playa|    0.0| 8300.0|21500.0|    0.0|29800.0|
+-------------+-------+-------+-------+-------+-------+



## Sección 4. Funciones de fecha

### Ejemplo 4.1 · Plazos de cobro de las facturas de una consultora

**Enunciado.** Una consultora emite facturas con un plazo de pago de **30 días**. El departamento financiero quiere saber, para cada factura, el año y el mes de emisión, el periodo contable (`yyyy-MM`), la fecha de vencimiento, cuántos días tardó en cobrarse y si se pagó tarde. Una factura todavía no se ha cobrado.

In [9]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

facturas = spark.createDataFrame(
    [("F-101", "2026-01-15", "2026-02-10", 3200.0),
     ("F-102", "2026-01-28", "2026-03-20", 1450.0),
     ("F-103", "2026-02-03", "2026-02-25", 980.0),
     ("F-104", "2026-02-17", None, 2750.0)],
    "factura STRING, emision STRING, pago STRING, importe DOUBLE",
)

con_fechas = (
    facturas
    .withColumn("emision", F.to_date("emision"))
    .withColumn("pago", F.to_date("pago"))
    .withColumn("anio", F.year("emision"))
    .withColumn("mes", F.month("emision"))
    .withColumn("periodo", F.date_format("emision", "yyyy-MM"))
    .withColumn("vencimiento", F.date_add("emision", 30))
    .withColumn("dias_cobro", F.datediff("pago", "emision"))
    .withColumn("pagada_tarde", col("pago") > col("vencimiento"))
)
con_fechas.printSchema()
con_fechas.drop("importe").show()

root
 |-- factura: string (nullable = true)
 |-- emision: date (nullable = true)
 |-- pago: date (nullable = true)
 |-- importe: double (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- periodo: string (nullable = true)
 |-- vencimiento: date (nullable = true)
 |-- dias_cobro: integer (nullable = true)
 |-- pagada_tarde: boolean (nullable = true)



+-------+----------+----------+----+---+-------+-----------+----------+------------+
|factura|   emision|      pago|anio|mes|periodo|vencimiento|dias_cobro|pagada_tarde|
+-------+----------+----------+----+---+-------+-----------+----------+------------+
|  F-101|2026-01-15|2026-02-10|2026|  1|2026-01| 2026-02-14|        26|       false|
|  F-102|2026-01-28|2026-03-20|2026|  1|2026-01| 2026-02-27|        51|        true|
|  F-103|2026-02-03|2026-02-25|2026|  2|2026-02| 2026-03-05|        22|       false|
|  F-104|2026-02-17|      NULL|2026|  2|2026-02| 2026-03-19|      NULL|        NULL|
+-------+----------+----------+----+---+-------+-----------+----------+------------+



### Ejemplo 4.2 · Reservas de un restaurante por mes y turno

**Enunciado.** Un grupo de restauración exporta las reservas con la fecha y la hora en formato europeo: `dd/MM/yyyy HH:mm`. Quiere clasificar cada reserva en **comida** (antes de las 17:00) o **cena**, saber si cae en **fin de semana** y, por mes y turno, el número de reservas, de comensales y el gasto medio por comensal.

In [10]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

reservas = spark.createDataFrame(
    [("R-1", "03/10/2026 18:45", 2, 64.0), ("R-2", "11/10/2026 21:10", 4, 152.0),
     ("R-3", "17/10/2026 14:30", 2, 58.0), ("R-4", "02/11/2026 20:00", 6, 231.0),
     ("R-5", "14/11/2026 13:15", 3, 96.0), ("R-6", "15/11/2026 22:05", 2, 71.0),
     ("R-7", "28/11/2026 21:30", 5, 188.0)],
    "reserva STRING, fecha_hora STRING, comensales INT, importe DOUBLE",
)

limpias = (
    reservas
    .withColumn("fecha_hora", F.to_timestamp("fecha_hora", "dd/MM/yyyy HH:mm"))
    .withColumn("fecha", F.to_date("fecha_hora"))
    .withColumn("hora", F.hour("fecha_hora"))
    .withColumn("turno", F.when(col("hora") < 17, "comida").otherwise("cena"))
    .withColumn("dia_semana", F.dayofweek("fecha"))
    .withColumn("fin_de_semana", col("dia_semana").isin(1, 7))
)
limpias.select("reserva", "fecha_hora", "fecha", "hora", "turno", "dia_semana", "fin_de_semana").show()

(limpias
 .groupBy(F.date_format("fecha", "yyyy-MM").alias("mes"), "turno")
 .agg(F.count("*").alias("reservas"),
      F.sum("comensales").alias("comensales"),
      F.round(F.sum("importe") / F.sum("comensales"), 2).alias("gasto_por_comensal"))
 .orderBy("mes", "turno")
 .show())

+-------+-------------------+----------+----+------+----------+-------------+
|reserva|         fecha_hora|     fecha|hora| turno|dia_semana|fin_de_semana|
+-------+-------------------+----------+----+------+----------+-------------+
|    R-1|2026-10-03 18:45:00|2026-10-03|  18|  cena|         7|         true|
|    R-2|2026-10-11 21:10:00|2026-10-11|  21|  cena|         1|         true|
|    R-3|2026-10-17 14:30:00|2026-10-17|  14|comida|         7|         true|
|    R-4|2026-11-02 20:00:00|2026-11-02|  20|  cena|         2|        false|
|    R-5|2026-11-14 13:15:00|2026-11-14|  13|comida|         7|         true|
|    R-6|2026-11-15 22:05:00|2026-11-15|  22|  cena|         1|         true|
|    R-7|2026-11-28 21:30:00|2026-11-28|  21|  cena|         7|         true|
+-------+-------------------+----------+----+------+----------+-------------+



+-------+------+--------+----------+------------------+
|    mes| turno|reservas|comensales|gasto_por_comensal|
+-------+------+--------+----------+------------------+
|2026-10|  cena|       2|         6|              36.0|
|2026-10|comida|       1|         2|              29.0|
|2026-11|  cena|       3|        13|             37.69|
|2026-11|comida|       1|         3|              32.0|
+-------+------+--------+----------+------------------+



## Sección 5. Joins

### Ejemplo 5.1 · Enriquecer los pedidos de una tienda de muebles con los datos del cliente

**Enunciado.** Una tienda de muebles para empresas guarda los pedidos con el código del cliente, pero el informe comercial necesita su nombre y su ciudad, que están en otra tabla. Se unen por `id_cliente`.

In [11]:
from pyspark.sql.functions import col

clientes = spark.createDataFrame(
    [("C-1", "Interiores Sol", "Valencia"), ("C-2", "Hotel Brisa", "Alicante"),
     ("C-3", "Oficinas Meridiano", "Madrid")],
    "id_cliente STRING, cliente STRING, ciudad STRING",
)
pedidos = spark.createDataFrame(
    [("PD-10", "C-1", "Sofá 3 plazas", 890.0), ("PD-11", "C-2", "Mesa de comedor", 640.0),
     ("PD-12", "C-1", "Lámpara de pie", 120.0), ("PD-13", "C-2", "Silla tapizada", 85.0)],
    "pedido STRING, id_cliente STRING, producto STRING, importe DOUBLE",
)

enriquecidos = pedidos.join(clientes, on="id_cliente", how="inner")
enriquecidos.printSchema()
enriquecidos.select("pedido", "cliente", "ciudad", "producto", "importe").orderBy("pedido").show()

root
 |-- id_cliente: string (nullable = true)
 |-- pedido: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- importe: double (nullable = true)
 |-- cliente: string (nullable = true)
 |-- ciudad: string (nullable = true)



+------+--------------+--------+---------------+-------+
|pedido|       cliente|  ciudad|       producto|importe|
+------+--------------+--------+---------------+-------+
| PD-10|Interiores Sol|Valencia|  Sofá 3 plazas|  890.0|
| PD-11|   Hotel Brisa|Alicante|Mesa de comedor|  640.0|
| PD-12|Interiores Sol|Valencia| Lámpara de pie|  120.0|
| PD-13|   Hotel Brisa|Alicante| Silla tapizada|   85.0|
+------+--------------+--------+---------------+-------+



### Ejemplo 5.2 · Los cuatro joins clásicos: catálogo y ventas

**Enunciado.** Una tienda de pequeño electrodoméstico compara su catálogo con las ventas del día. Hay un producto del catálogo sin ventas (la tostadora, `A2`) y una venta de un producto que no está en el catálogo (`Z9`). Se ejecutan los cuatro tipos de join en un bucle para ver qué devuelve cada uno.

In [12]:
from pyspark.sql.functions import col

catalogo = spark.createDataFrame(
    [("A1", "Cafetera"), ("A2", "Tostadora"), ("A3", "Batidora")],
    "sku STRING, producto STRING",
)
ventas = spark.createDataFrame(
    [("A1", 5), ("A1", 2), ("A3", 1), ("Z9", 4)],
    "sku STRING, unidades INT",
)

for tipo in ["inner", "left", "right", "full"]:
    resultado = catalogo.join(ventas, on="sku", how=tipo).orderBy("sku", "unidades")
    print(f"--- {tipo}: {resultado.count()} filas ---")
    resultado.show()

--- inner: 3 filas ---


+---+--------+--------+
|sku|producto|unidades|
+---+--------+--------+
| A1|Cafetera|       2|
| A1|Cafetera|       5|
| A3|Batidora|       1|
+---+--------+--------+



--- left: 4 filas ---


+---+---------+--------+
|sku| producto|unidades|
+---+---------+--------+
| A1| Cafetera|       2|
| A1| Cafetera|       5|
| A2|Tostadora|    NULL|
| A3| Batidora|       1|
+---+---------+--------+



--- right: 4 filas ---


+---+--------+--------+
|sku|producto|unidades|
+---+--------+--------+
| A1|Cafetera|       2|
| A1|Cafetera|       5|
| A3|Batidora|       1|
| Z9|    NULL|       4|
+---+--------+--------+



--- full: 5 filas ---


+---+---------+--------+
|sku| producto|unidades|
+---+---------+--------+
| A1| Cafetera|       2|
| A1| Cafetera|       5|
| A2|Tostadora|    NULL|
| A3| Batidora|       1|
| Z9|     NULL|       4|
+---+---------+--------+



### Ejemplo 5.3 · Clientes sin facturas y facturas huérfanas

**Enunciado.** Una gestoría que lleva la contabilidad de pequeños negocios quiere saber qué clientes han tenido actividad este mes (tienen alguna factura), cuáles no (para llamarlos) y si hay **facturas que apuntan a un cliente que no existe**, lo que indicaría un error en la carga de datos.

In [13]:
from pyspark.sql.functions import col

clientes = spark.createDataFrame(
    [("C-1", "Gimnasio Atlas"), ("C-2", "Clínica Delta"), ("C-3", "Bar La Ribera"),
     ("C-4", "Taller Moreno")],
    "id_cliente STRING, cliente STRING",
)
facturas = spark.createDataFrame(
    [("F-1", "C-1", 300.0), ("F-2", "C-1", 120.0), ("F-3", "C-3", 75.0), ("F-4", "C-9", 410.0)],
    "factura STRING, id_cliente STRING, importe DOUBLE",
)

print("1) Clientes que tienen alguna factura (left_semi)")
clientes.join(facturas, on="id_cliente", how="left_semi").orderBy("id_cliente").show()

print("2) Clientes sin facturas (left_anti)")
clientes.join(facturas, on="id_cliente", how="left_anti").orderBy("id_cliente").show()

print("3) Facturas de clientes que no existen (control de integridad)")
huerfanas = facturas.join(clientes, on="id_cliente", how="left_anti")
huerfanas.show()
print("Facturas huérfanas:", huerfanas.count())

1) Clientes que tienen alguna factura (left_semi)


+----------+--------------+
|id_cliente|       cliente|
+----------+--------------+
|       C-1|Gimnasio Atlas|
|       C-3| Bar La Ribera|
+----------+--------------+

2) Clientes sin facturas (left_anti)


+----------+-------------+
|id_cliente|      cliente|
+----------+-------------+
|       C-2|Clínica Delta|
|       C-4|Taller Moreno|
+----------+-------------+

3) Facturas de clientes que no existen (control de integridad)


+----------+-------+-------+
|id_cliente|factura|importe|
+----------+-------+-------+
|       C-9|    F-4|  410.0|
+----------+-------+-------+



Facturas huérfanas: 1


### Ejemplo 5.4 · Claves con distinto nombre y la misma tabla dos veces

**Enunciado.** Una empresa de logística registra los traslados de palets entre sus almacenes. Cada traslado tiene un almacén de **origen** y uno de **destino**, y los dos son códigos de la **misma** tabla de almacenes. El informe tiene que mostrar el nombre de la ciudad de origen y el de la ciudad de destino.

In [14]:
from pyspark.sql.functions import col

almacenes = spark.createDataFrame(
    [("MAD", "Madrid", "Getafe"), ("BCN", "Barcelona", "El Prat"), ("VLC", "Valencia", "Riba-roja")],
    "codigo STRING, ciudad STRING, poligono STRING",
)
traslados = spark.createDataFrame(
    [("TR-1", "MAD", "BCN", 22), ("TR-2", "BCN", "VLC", 8), ("TR-3", "VLC", "MAD", 15)],
    "traslado STRING, origen STRING, destino STRING, palets INT",
)

t = traslados.alias("t")
o = almacenes.alias("o")
d = almacenes.alias("d")

informe = (
    t.join(o, col("t.origen") == col("o.codigo"), "inner")
     .join(d, col("t.destino") == col("d.codigo"), "inner")
     .select(
         col("t.traslado"),
         col("o.ciudad").alias("ciudad_origen"),
         col("d.ciudad").alias("ciudad_destino"),
         col("t.palets"),
     )
     .orderBy("traslado")
)
informe.show()

+--------+-------------+--------------+------+
|traslado|ciudad_origen|ciudad_destino|palets|
+--------+-------------+--------------+------+
|    TR-1|       Madrid|     Barcelona|    22|
|    TR-2|    Barcelona|      Valencia|     8|
|    TR-3|     Valencia|        Madrid|    15|
+--------+-------------+--------------+------+



### Ejemplo 5.5 · Join con una tabla pequeña y agregación posterior

**Enunciado.** Un supermercado online calcula la facturación con IVA de cada categoría. Las líneas de venta solo tienen el código de categoría; el nombre y el tipo de IVA están en una tabla de **tres filas**. Como esa tabla es diminuta, se pide un **broadcast join**.

In [15]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

categorias = spark.createDataFrame(
    [("ELEC", "Electrónica", 0.21), ("ALIM", "Alimentación", 0.10), ("LIBR", "Libros", 0.04)],
    "cod_categoria STRING, categoria STRING, iva DOUBLE",
)
lineas = spark.createDataFrame(
    [("L1", "ELEC", 2, 199.0), ("L2", "ALIM", 10, 3.2), ("L3", "LIBR", 3, 18.5),
     ("L4", "ELEC", 1, 649.0), ("L5", "ALIM", 25, 1.9), ("L6", "LIBR", 1, 32.0)],
    "linea STRING, cod_categoria STRING, unidades INT, precio DOUBLE",
)

por_categoria = (
    lineas
    .join(F.broadcast(categorias), on="cod_categoria", how="inner")
    .withColumn("base", col("unidades") * col("precio"))
    .withColumn("total", col("base") * (1 + col("iva")))
    .groupBy("categoria")
    .agg(F.count("*").alias("lineas"),
         F.round(F.sum("base"), 2).alias("base"),
         F.round(F.sum("total"), 2).alias("total_con_iva"))
    .orderBy(col("total_con_iva").desc())
)
por_categoria.show()

+------------+------+------+-------------+
|   categoria|lineas|  base|total_con_iva|
+------------+------+------+-------------+
| Electrónica|     2|1047.0|      1266.87|
|      Libros|     2|  87.5|         91.0|
|Alimentación|     2|  79.5|        87.45|
+------------+------+------+-------------+



## Sección 6. Funciones de ventana

### Ejemplo 6.1 · Quedarse con la versión más reciente de cada cliente

**Enunciado.** El CRM de una distribuidora envía **todas** las modificaciones de los datos de sus clientes: cada vez que un cliente cambia de email o de ciudad llega una fila nueva. Para cargar la tabla de clientes hay que quedarse solo con la **versión vigente**, la de la fecha más reciente.

In [16]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

cambios = spark.createDataFrame(
    [("CL-7", "2026-09-02", "ana.gil@correo.es", "Zaragoza"),
     ("CL-7", "2026-10-15", "ana.gil@empresa.es", "Zaragoza"),
     ("CL-8", "2026-08-20", "pablo@tienda.es", "Lleida"),
     ("CL-7", "2026-11-03", "ana.gil@empresa.es", "Huesca"),
     ("CL-9", "2026-10-01", "marta@estudio.es", "Soria"),
     ("CL-8", "2026-11-10", "pablo.rey@tienda.es", "Lleida")],
    "id_cliente STRING, actualizado STRING, email STRING, ciudad STRING",
).withColumn("actualizado", F.to_date("actualizado"))

por_cliente = Window.partitionBy("id_cliente").orderBy(col("actualizado").desc())

numeradas = cambios.withColumn("version", F.row_number().over(por_cliente))
numeradas.orderBy("id_cliente", "version").show()

vigentes = numeradas.filter(col("version") == 1).drop("version").orderBy("id_cliente")
vigentes.show()
print("Registros:", cambios.count(), "| clientes:", vigentes.count())

+----------+-----------+-------------------+--------+-------+
|id_cliente|actualizado|              email|  ciudad|version|
+----------+-----------+-------------------+--------+-------+
|      CL-7| 2026-11-03| ana.gil@empresa.es|  Huesca|      1|
|      CL-7| 2026-10-15| ana.gil@empresa.es|Zaragoza|      2|
|      CL-7| 2026-09-02|  ana.gil@correo.es|Zaragoza|      3|
|      CL-8| 2026-11-10|pablo.rey@tienda.es|  Lleida|      1|
|      CL-8| 2026-08-20|    pablo@tienda.es|  Lleida|      2|
|      CL-9| 2026-10-01|   marta@estudio.es|   Soria|      1|
+----------+-----------+-------------------+--------+-------+



+----------+-----------+-------------------+------+
|id_cliente|actualizado|              email|ciudad|
+----------+-----------+-------------------+------+
|      CL-7| 2026-11-03| ana.gil@empresa.es|Huesca|
|      CL-8| 2026-11-10|pablo.rey@tienda.es|Lleida|
|      CL-9| 2026-10-01|   marta@estudio.es| Soria|
+----------+-----------+-------------------+------+



Registros: 6 | clientes: 3


### Ejemplo 6.2 · Ranking de productos por categoría con empates

**Enunciado.** Una tienda de deporte quiere el ranking de productos más vendidos dentro de cada categoría. Hay empates: la bota de montaña y la sandalia de trekking han vendido lo mismo, y también la chaqueta y las mallas. Se comparan las tres funciones de ranking y se obtiene el **top 2** de cada categoría sin dejar fuera a los empatados.

In [17]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

ventas = spark.createDataFrame(
    [("Calzado", "Zapatilla running", 540), ("Calzado", "Bota montaña", 410),
     ("Calzado", "Sandalia trekking", 410), ("Calzado", "Zapatilla trail", 230),
     ("Textil", "Chaqueta cortavientos", 300), ("Textil", "Camiseta técnica", 720),
     ("Textil", "Mallas", 300)],
    "categoria STRING, producto STRING, unidades INT",
)

ventana = Window.partitionBy("categoria").orderBy(col("unidades").desc())
con_desempate = Window.partitionBy("categoria").orderBy(col("unidades").desc(), col("producto"))

ranking = (
    ventas
    .withColumn("row_number", F.row_number().over(con_desempate))
    .withColumn("rank", F.rank().over(ventana))
    .withColumn("dense_rank", F.dense_rank().over(ventana))
    .orderBy("categoria", "row_number", "producto")
)
ranking.show(truncate=False)

print("Top 2 por categoría (con empates):")
(ranking
 .filter(col("dense_rank") <= 2)
 .select("categoria", "producto", "unidades")
 .orderBy("categoria", col("unidades").desc(), "producto")
 .show(truncate=False))

+---------+---------------------+--------+----------+----+----------+
|categoria|producto             |unidades|row_number|rank|dense_rank|
+---------+---------------------+--------+----------+----+----------+
|Calzado  |Zapatilla running    |540     |1         |1   |1         |
|Calzado  |Bota montaña         |410     |2         |2   |2         |
|Calzado  |Sandalia trekking    |410     |3         |2   |2         |
|Calzado  |Zapatilla trail      |230     |4         |4   |3         |
|Textil   |Camiseta técnica     |720     |1         |1   |1         |
|Textil   |Chaqueta cortavientos|300     |2         |2   |2         |
|Textil   |Mallas               |300     |3         |2   |2         |
+---------+---------------------+--------+----------+----+----------+

Top 2 por categoría (con empates):


+---------+---------------------+--------+
|categoria|producto             |unidades|
+---------+---------------------+--------+
|Calzado  |Zapatilla running    |540     |
|Calzado  |Bota montaña         |410     |
|Calzado  |Sandalia trekking    |410     |
|Textil   |Camiseta técnica     |720     |
|Textil   |Chaqueta cortavientos|300     |
|Textil   |Mallas               |300     |
+---------+---------------------+--------+



### Ejemplo 6.3 · Variación diaria, acumulado y media móvil de unas panaderías

**Enunciado.** Una cadena de panaderías analiza las ventas diarias de dos locales. Para cada día quiere: las ventas del día anterior, la variación respecto a ese día, las ventas acumuladas desde el inicio de la semana y la media de los últimos 3 días.

In [18]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

diarias = spark.createDataFrame(
    [("Panadería Sol", "2026-11-02", 410.0), ("Panadería Sol", "2026-11-03", 385.0),
     ("Panadería Sol", "2026-11-04", 462.0), ("Panadería Sol", "2026-11-05", 520.0),
     ("Panadería Luna", "2026-11-02", 298.0), ("Panadería Luna", "2026-11-03", 341.0),
     ("Panadería Luna", "2026-11-04", 317.0)],
    "tienda STRING, fecha STRING, ventas DOUBLE",
).withColumn("fecha", F.to_date("fecha"))

por_tienda = Window.partitionBy("tienda").orderBy("fecha")
acumulado = por_tienda.rowsBetween(Window.unboundedPreceding, Window.currentRow)
ultimos_3 = por_tienda.rowsBetween(-2, Window.currentRow)

analisis = (
    diarias
    .withColumn("dia_anterior", F.lag("ventas", 1).over(por_tienda))
    .withColumn("variacion", F.round(col("ventas") - col("dia_anterior"), 2))
    .withColumn("acumulado", F.sum("ventas").over(acumulado))
    .withColumn("media_3_dias", F.round(F.avg("ventas").over(ultimos_3), 2))
    .orderBy("tienda", "fecha")
)
analisis.show()

+--------------+----------+------+------------+---------+---------+------------+
|        tienda|     fecha|ventas|dia_anterior|variacion|acumulado|media_3_dias|
+--------------+----------+------+------------+---------+---------+------------+
|Panadería Luna|2026-11-02| 298.0|        NULL|     NULL|    298.0|       298.0|
|Panadería Luna|2026-11-03| 341.0|       298.0|     43.0|    639.0|       319.5|
|Panadería Luna|2026-11-04| 317.0|       341.0|    -24.0|    956.0|      318.67|
| Panadería Sol|2026-11-02| 410.0|        NULL|     NULL|    410.0|       410.0|
| Panadería Sol|2026-11-03| 385.0|       410.0|    -25.0|    795.0|       397.5|
| Panadería Sol|2026-11-04| 462.0|       385.0|     77.0|   1257.0|       419.0|
| Panadería Sol|2026-11-05| 520.0|       462.0|     58.0|   1777.0|      455.67|
+--------------+----------+------+------------+---------+---------+------------+



## Sección 7. Spark SQL

### Ejemplo 7.1 · Ingresos por plan de una plataforma de streaming

**Enunciado.** Una plataforma de vídeo bajo demanda quiere los ingresos mensuales por plan en España, solo de los planes que generan al menos 10 € al mes. El analista lo escribe en SQL y después sigue con la API de DataFrames para calcular el ingreso anual y comprobar si cada plan supera el objetivo de 300 €.

In [19]:
from pyspark.sql import functions as F

suscripciones = spark.createDataFrame(
    [("U-1", "Premium", "España", 14.99), ("U-2", "Básico", "España", 7.99),
     ("U-3", "Premium", "México", 9.99), ("U-4", "Familiar", "España", 19.99),
     ("U-5", "Básico", "México", 4.99), ("U-6", "Premium", "España", 14.99),
     ("U-7", "Familiar", "México", 12.99), ("U-8", "Premium", "México", 9.99)],
    "usuario STRING, plan STRING, pais STRING, cuota DOUBLE",
)
suscripciones.createOrReplaceTempView("suscripciones")

por_plan = spark.sql("""
    SELECT plan,
           COUNT(*)              AS usuarios,
           ROUND(SUM(cuota), 2)  AS ingresos_mes
    FROM suscripciones
    WHERE pais = 'España'
    GROUP BY plan
    HAVING SUM(cuota) >= 10
    ORDER BY ingresos_mes DESC
""")
print("Tipo del resultado:", type(por_plan).__name__)
por_plan.show()

(por_plan
 .withColumn("ingresos_anio", F.round(F.col("ingresos_mes") * 12, 2))
 .selectExpr("plan", "ingresos_anio", "ingresos_anio > 300 AS objetivo_cumplido")
 .show())

Tipo del resultado: DataFrame


+--------+--------+------------+
|    plan|usuarios|ingresos_mes|
+--------+--------+------------+
| Premium|       2|       29.98|
|Familiar|       1|       19.99|
+--------+--------+------------+



+--------+-------------+-----------------+
|    plan|ingresos_anio|objetivo_cumplido|
+--------+-------------+-----------------+
| Premium|       359.76|             true|
|Familiar|       239.88|            false|
+--------+-------------+-----------------+



### Ejemplo 7.2 · Saldo de las cuentas con una CTE y una función de ventana

**Enunciado.** Un banco recibe los movimientos de sus cuentas con la fecha como texto. Necesita el **saldo acumulado** después de cada movimiento y marcar los momentos en que una cuenta queda **en descubierto** (saldo negativo). El equipo, que viene de SQL Server, lo resuelve con CTE y una función de ventana.

In [20]:
movimientos = spark.createDataFrame(
    [("ES-01", "2026-11-03", -45.20), ("ES-01", "2026-11-05", 1250.00),
     ("ES-01", "2026-11-09", -320.00), ("ES-02", "2026-11-02", 600.00),
     ("ES-02", "2026-11-06", -75.50), ("ES-02", "2026-11-10", -980.00)],
    "cuenta STRING, fecha STRING, importe DOUBLE",
)
movimientos.createOrReplaceTempView("movimientos")

spark.sql("""
    WITH tipados AS (
        SELECT cuenta,
               CAST(fecha AS DATE) AS fecha,
               importe
        FROM movimientos
    ),
    con_saldo AS (
        SELECT cuenta, fecha, importe,
               SUM(importe) OVER (PARTITION BY cuenta ORDER BY fecha) AS saldo
        FROM tipados
    )
    SELECT cuenta, fecha, importe, ROUND(saldo, 2) AS saldo,
           saldo < 0 AS en_descubierto
    FROM con_saldo
    ORDER BY cuenta, fecha
""").show()

+------+----------+-------+------+--------------+
|cuenta|     fecha|importe| saldo|en_descubierto|
+------+----------+-------+------+--------------+
| ES-01|2026-11-03|  -45.2| -45.2|          true|
| ES-01|2026-11-05| 1250.0|1204.8|         false|
| ES-01|2026-11-09| -320.0| 884.8|         false|
| ES-02|2026-11-02|  600.0| 600.0|         false|
| ES-02|2026-11-06|  -75.5| 524.5|         false|
| ES-02|2026-11-10| -980.0|-455.5|          true|
+------+----------+-------+------+--------------+



## Sección 8. Escribir datos

### Ejemplo 8.1 · Guardar lecturas de sensores en Parquet y en CSV

**Enunciado.** Una empresa de climatización guarda las lecturas diarias de sus sensores. Se escriben en Parquet y en CSV, se vuelven a leer para comparar qué se conserva, se comprueba qué pasa al escribir sin indicar el modo y, por último, se añaden filas con `append`.

In [21]:
from datetime import date

lecturas = spark.createDataFrame(
    [("SEN-01", date(2026, 11, 10), 21.4, True), ("SEN-02", date(2026, 11, 10), 23.9, True),
     ("SEN-03", date(2026, 11, 10), None, False)],
    "sensor STRING, fecha DATE, temperatura DOUBLE, activo BOOLEAN",
)

# 1. Escribir en Parquet y en CSV
lecturas.write.mode("overwrite").parquet(f"{RUTA}/lecturas_parquet")
lecturas.write.mode("overwrite").option("header", True).csv(f"{RUTA}/lecturas_csv")

# 2. Leer de nuevo y comparar los esquemas
print("--- Leído desde Parquet ---")
spark.read.parquet(f"{RUTA}/lecturas_parquet").printSchema()
print("--- Leído desde CSV ---")
spark.read.option("header", True).csv(f"{RUTA}/lecturas_csv").printSchema()

# 3. Modo por defecto (errorifexists): falla si la carpeta ya existe
try:
    lecturas.write.parquet(f"{RUTA}/lecturas_parquet")
except Exception:
    print("No se ha escrito: la carpeta ya existe y el modo es errorifexists")

# 4. Modo append: añade filas a lo que ya había
lecturas.write.mode("append").parquet(f"{RUTA}/lecturas_parquet")
print("Filas tras append:", spark.read.parquet(f"{RUTA}/lecturas_parquet").count())

--- Leído desde Parquet ---
root
 |-- sensor: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- activo: boolean (nullable = true)

--- Leído desde CSV ---


root
 |-- sensor: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- temperatura: string (nullable = true)
 |-- activo: string (nullable = true)

No se ha escrito: la carpeta ya existe y el modo es errorifexists


Filas tras append: 6


### Ejemplo 8.2 · Pedidos particionados por mes y región

**Enunciado.** Un mayorista guarda el histórico de pedidos para que los analistas consulten un mes concreto sin leer todo el histórico. Se escriben particionados por **mes** y **región**, se comprueba la estructura de carpetas creada y se lee solo noviembre.

In [22]:
from pathlib import Path

from pyspark.sql import functions as F

pedidos = spark.createDataFrame(
    [("PE-1", "2026-10-28", "Norte", 180.0), ("PE-2", "2026-10-30", "Sur", 95.0),
     ("PE-3", "2026-11-02", "Norte", 240.0), ("PE-4", "2026-11-05", "Norte", 60.0),
     ("PE-5", "2026-11-07", "Sur", 310.0), ("PE-6", "2026-11-09", "Centro", 125.0)],
    "pedido STRING, fecha STRING, region STRING, importe DOUBLE",
).withColumn("mes", F.date_format(F.to_date("fecha"), "yyyy-MM"))

(pedidos.write
 .mode("overwrite")
 .partitionBy("mes", "region")
 .parquet(f"{RUTA}/pedidos_particionados"))

base = Path(f"{RUTA}/pedidos_particionados")
for carpeta_mes in sorted(p for p in base.iterdir() if p.is_dir()):
    regiones = sorted(p.name for p in carpeta_mes.iterdir() if p.is_dir())
    print(carpeta_mes.name, "->", regiones)

noviembre = spark.read.parquet(f"{RUTA}/pedidos_particionados").filter(F.col("mes") == "2026-11")
noviembre.orderBy("pedido").show()

mes=2026-10 -> ['region=Norte', 'region=Sur']
mes=2026-11 -> ['region=Centro', 'region=Norte', 'region=Sur']


+------+----------+-------+-------+------+
|pedido|     fecha|importe|    mes|region|
+------+----------+-------+-------+------+
|  PE-3|2026-11-02|  240.0|2026-11| Norte|
|  PE-4|2026-11-05|   60.0|2026-11| Norte|
|  PE-5|2026-11-07|  310.0|2026-11|   Sur|
|  PE-6|2026-11-09|  125.0|2026-11|Centro|
+------+----------+-------+-------+------+



### Ejemplo 8.3 · Una tabla de tarifas de envío en el catálogo

**Enunciado.** El equipo de e-commerce mantiene una tabla con el coste de envío por zona, que usan otros notebooks y los informes de negocio. Se crea la tabla, se añade una zona nueva (Portugal), se comprueba que está registrada en el catálogo y se consulta con la API de DataFrames y con SQL.

In [23]:
tarifas = spark.createDataFrame(
    [("ES-PEN", "Península", 4.95), ("ES-BAL", "Baleares", 8.50), ("ES-CAN", "Canarias", 12.90)],
    "zona STRING, descripcion STRING, coste_envio DOUBLE",
)

# 1. Crear (o sustituir) la tabla
tarifas.write.mode("overwrite").saveAsTable(f"{BD}.tarifas_envio")

# 2. Añadir filas nuevas
nuevas = spark.createDataFrame([("PT-CON", "Portugal", 9.75)], tarifas.schema)
nuevas.write.mode("append").saveAsTable(f"{BD}.tarifas_envio")

# 3. Comprobar que la tabla está en el catálogo
tablas = [fila.tableName for fila in spark.sql(f"SHOW TABLES IN {BD}").collect()]
print("¿Está registrada la tabla?", "tarifas_envio" in tablas)

# 4. Leer la tabla con la API de DataFrames y con SQL
tabla = spark.table(f"{BD}.tarifas_envio")
print("Filas en la tabla:", tabla.count())
tabla.orderBy("coste_envio").show()

spark.sql(f"""
    SELECT zona, coste_envio
    FROM {BD}.tarifas_envio
    WHERE coste_envio > 9
    ORDER BY coste_envio DESC
""").show()

¿Está registrada la tabla? True
Filas en la tabla: 4


+------+-----------+-----------+
|  zona|descripcion|coste_envio|
+------+-----------+-----------+
|ES-PEN|  Península|       4.95|
|ES-BAL|   Baleares|        8.5|
|PT-CON|   Portugal|       9.75|
|ES-CAN|   Canarias|       12.9|
+------+-----------+-----------+

+------+-----------+
|  zona|coste_envio|
+------+-----------+
|ES-CAN|       12.9|
|PT-CON|       9.75|
+------+-----------+



## Sección 9. Un pipeline completo: de los ficheros a las tablas finales

### Ejemplo 9.1 · Ventas de una tienda online de electrodomésticos

**Enunciado.** Una tienda online de electrodomésticos recibe cada noche dos ficheros:

- `raw_lineas_pedido.csv`: las líneas de pedido exportadas de la web, con datos sucios: SKU en minúsculas y con espacios, unidades escritas en letra, una línea duplicada, una línea sin fecha y una venta de un SKU que no existe.
- `raw_catalogo.json`: el catálogo del ERP, con el nombre y la categoría de cada producto.

Se quieren dos tablas finales para el equipo de negocio:

1. `ventas_categoria_mes`: unidades y facturación por categoría y mes.
2. `top_producto_categoria`: el producto con más facturación de cada categoría.

Además, el pipeline debe informar de las líneas con un SKU que no está en el catálogo.

In [24]:
from pathlib import Path

from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window

# 0. Ficheros de origen: exportación de la tienda online y catálogo del ERP
Path(f"{RUTA}/raw_lineas_pedido.csv").write_text(
    "id_linea,fecha,sku,unidades,precio\n"
    "L-001,2026-10-04,frig-01,1,649.00\n"
    "L-002,2026-10-04, LAV-02 ,2,429.00\n"
    "L-003,2026-10-11,MIC-03,3,89.90\n"
    "L-004,2026-10-11,MIC-03,tres,89.90\n"
    "L-005,2026-10-19,LAV-02,1,429.00\n"
    "L-002,2026-10-04, LAV-02 ,2,429.00\n"
    "L-006,2026-11-02,FRIG-01,2,649.00\n"
    "L-007,2026-11-02,ASP-04,4,159.00\n"
    "L-008,2026-11-09,MIC-03,5,89.90\n"
    "L-009,2026-11-15,XXX-99,1,99.00\n"
    "L-010,2026-11-20,ASP-04,1,159.00\n"
    "L-011,,LAV-02,1,429.00\n",
    encoding="utf-8",
)
Path(f"{RUTA}/raw_catalogo.json").write_text(
    '{"sku": "FRIG-01", "producto": "Frigorífico combi", "categoria": "Gran electrodoméstico"}\n'
    '{"sku": "LAV-02", "producto": "Lavadora 9 kg", "categoria": "Gran electrodoméstico"}\n'
    '{"sku": "MIC-03", "producto": "Microondas", "categoria": "Pequeño electrodoméstico"}\n'
    '{"sku": "ASP-04", "producto": "Aspiradora sin cable", "categoria": "Pequeño electrodoméstico"}\n'
    '{"sku": "HOR-05", "producto": "Horno multifunción", "categoria": "Gran electrodoméstico"}\n',
    encoding="utf-8",
)

# 1. EXTRACCIÓN
lineas_raw = spark.read.csv(f"{RUTA}/raw_lineas_pedido.csv", header=True)
catalogo = spark.read.json(f"{RUTA}/raw_catalogo.json")
print("Líneas leídas:", lineas_raw.count(), "| productos en catálogo:", catalogo.count())

# 2. LIMPIEZA
lineas = (
    lineas_raw
    .dropDuplicates(["id_linea"])
    .withColumn("sku", F.upper(F.trim(col("sku"))))
    .withColumn("fecha", F.to_date("fecha"))
    .withColumn("unidades", col("unidades").try_cast("int"))
    .withColumn("precio", col("precio").try_cast("double"))
    .filter(col("fecha").isNotNull() & col("unidades").isNotNull() & col("precio").isNotNull())
)
print("Líneas válidas:", lineas.count())

# 3. CONTROL DE INTEGRIDAD: productos que no están en el catálogo
desconocidas = lineas.join(catalogo, on="sku", how="left_anti")
print("Líneas con SKU desconocido:", [f.id_linea for f in desconocidas.collect()])

# 4. ENRIQUECIMIENTO
ventas = (
    lineas
    .join(F.broadcast(catalogo), on="sku", how="inner")
    .withColumn("importe", F.round(col("unidades") * col("precio"), 2))
    .withColumn("mes", F.date_format("fecha", "yyyy-MM"))
)

# 5. AGREGACIÓN: ventas por categoría y mes
por_categoria_mes = (
    ventas
    .groupBy("categoria", "mes")
    .agg(F.sum("unidades").alias("unidades"),
         F.round(F.sum("importe"), 2).alias("facturacion"))
    .orderBy("categoria", "mes")
)
por_categoria_mes.show(truncate=False)

# 6. RANKING: producto más vendido (en importe) de cada categoría
ventana = Window.partitionBy("categoria").orderBy(col("facturacion").desc())
top_productos = (
    ventas
    .groupBy("categoria", "producto")
    .agg(F.round(F.sum("importe"), 2).alias("facturacion"))
    .withColumn("puesto", F.dense_rank().over(ventana))
    .filter(col("puesto") == 1)
    .drop("puesto")
    .orderBy("categoria")
)
top_productos.show(truncate=False)

# 7. CARGA: tablas de la capa final
por_categoria_mes.write.mode("overwrite").saveAsTable(f"{BD}.ventas_categoria_mes")
top_productos.write.mode("overwrite").saveAsTable(f"{BD}.top_producto_categoria")
print("Filas en ventas_categoria_mes:", spark.table(f"{BD}.ventas_categoria_mes").count())
print("Filas en top_producto_categoria:", spark.table(f"{BD}.top_producto_categoria").count())

Líneas leídas: 12 | productos en catálogo: 5


Líneas válidas: 9


Líneas con SKU desconocido: ['L-009']


+------------------------+-------+--------+-----------+
|categoria               |mes    |unidades|facturacion|
+------------------------+-------+--------+-----------+
|Gran electrodoméstico   |2026-10|4       |1936.0     |
|Gran electrodoméstico   |2026-11|2       |1298.0     |
|Pequeño electrodoméstico|2026-10|3       |269.7      |
|Pequeño electrodoméstico|2026-11|10      |1244.5     |
+------------------------+-------+--------+-----------+



+------------------------+--------------------+-----------+
|categoria               |producto            |facturacion|
+------------------------+--------------------+-----------+
|Gran electrodoméstico   |Frigorífico combi   |1947.0     |
|Pequeño electrodoméstico|Aspiradora sin cable|795.0      |
+------------------------+--------------------+-----------+



Filas en ventas_categoria_mes: 4
Filas en top_producto_categoria: 2


## Cerrar la sesión

En local y en Docker se libera la sesión; en Databricks la gestiona la plataforma.

In [25]:
if ENTORNO != "databricks":
    spark.stop()